# Level 5: Oracles and the Deutsch-Jozsa Algorithm
## Qiskit Edition

In this notebook, you will:

1. Understand what a **quantum oracle** is
2. Learn the difference between **constant** and **balanced** functions
3. Implement the **Deutsch-Jozsa algorithm** — the first quantum speedup
4. See quantum interference produce a deterministic answer in ONE query

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

simulator = AerSimulator()
print("Ready!")

---
## 1. What Is a Quantum Oracle?

An **oracle** is a black-box function that we can query. In classical computing, we call it and look at the output. In quantum computing, we can query it in **superposition**!

The standard oracle model:

$$U_f |x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$$

where $\oplus$ is addition modulo 2 (XOR).

### Phase Oracle Trick
If we set $|y\rangle = |{-}\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$, then:

$$U_f |x\rangle|{-}\rangle = (-1)^{f(x)} |x\rangle|{-}\rangle$$

The function value gets "kicked back" as a **phase**! This is called the **phase kickback** trick.

---
## 2. The Deutsch-Jozsa Problem

**Promise**: You're given a function $f: \{0,1\}^n \to \{0,1\}$ that is either:
- **Constant**: $f(x) = 0$ for all $x$, OR $f(x) = 1$ for all $x$
- **Balanced**: $f(x) = 0$ for exactly half the inputs, $f(x) = 1$ for the other half

**Task**: Determine whether $f$ is constant or balanced.

**Classical**: Need up to $2^{n-1} + 1$ queries (worst case)

**Quantum**: Need exactly **1** query! This is an exponential speedup.

### The Algorithm:

1. Start with $|0\rangle^{\otimes n}|1\rangle$
2. Apply $H$ to all qubits
3. Apply the oracle $U_f$
4. Apply $H$ to the first $n$ qubits
5. Measure the first $n$ qubits

**Result**: If all zeros → **constant**. If any one is 1 → **balanced**.

In [ ]:
# First, let's build some oracle functions

def constant_zero_oracle(qc, input_qubits, output_qubit):
    """f(x) = 0 for all x. Do nothing."""
    pass

def constant_one_oracle(qc, input_qubits, output_qubit):
    """f(x) = 1 for all x. Flip the output."""
    qc.x(output_qubit)

def balanced_oracle_1(qc, input_qubits, output_qubit):
    """f(x) = x_0 (first bit). CNOT from input[0] to output."""
    qc.cx(input_qubits[0], output_qubit)

def balanced_oracle_2(qc, input_qubits, output_qubit):
    """f(x) = x_0 XOR x_1. Two CNOTs."""
    qc.cx(input_qubits[0], output_qubit)
    qc.cx(input_qubits[1], output_qubit)

print("Oracles defined!")

In [ ]:
def deutsch_jozsa(n_qubits, oracle):
    """Implement the Deutsch-Jozsa algorithm.
    
    n_qubits: number of input qubits
    oracle: function that builds the oracle circuit
    """
    n = n_qubits
    qc = QuantumCircuit(n + 1, n)  # n input qubits + 1 output qubit
    
    # Step 1: Initialize output qubit to |1>
    qc.x(n)
    
    # Step 2: Apply H to ALL qubits (input + output)
    qc.h(range(n + 1))
    qc.barrier()
    
    # Step 3: Apply the oracle
    oracle(qc, list(range(n)), n)
    qc.barrier()
    
    # Step 4: Apply H to input qubits only
    qc.h(range(n))
    
    # Step 5: Measure input qubits
    qc.measure(range(n), range(n))
    
    return qc

print("Deutsch-Jozsa algorithm defined!")

In [ ]:
# Test with the CONSTANT oracle (f(x) = 0)
qc_const = deutsch_jozsa(3, constant_zero_oracle)
print("Circuit for constant oracle f(x) = 0:")
display(qc_const.draw('mpl'))

result = simulator.run(qc_const, shots=1000).result().get_counts()
print(f"\nResults: {result}")
print("All zeros → CONSTANT ✓")

In [ ]:
# Test with the other CONSTANT oracle (f(x) = 1)
qc_const1 = deutsch_jozsa(3, constant_one_oracle)
result = simulator.run(qc_const1, shots=1000).result().get_counts()
print(f"Constant f(x)=1 results: {result}")
print("All zeros → CONSTANT ✓")

In [ ]:
# Test with BALANCED oracle 1: f(x) = x_0
qc_bal1 = deutsch_jozsa(3, balanced_oracle_1)
print("Circuit for balanced oracle f(x) = x₀:")
display(qc_bal1.draw('mpl'))

result = simulator.run(qc_bal1, shots=1000).result().get_counts()
print(f"\nResults: {result}")
print("NOT all zeros → BALANCED ✓")

In [ ]:
# Test with BALANCED oracle 2: f(x) = x_0 XOR x_1
qc_bal2 = deutsch_jozsa(3, balanced_oracle_2)
result = simulator.run(qc_bal2, shots=1000).result().get_counts()
print(f"Balanced f(x)=x₀⊕x₁ results: {result}")
print("NOT all zeros → BALANCED ✓")

In [ ]:
# Compare all results visually
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

oracles = [
    (constant_zero_oracle, "Constant (0)"),
    (constant_one_oracle, "Constant (1)"),
    (balanced_oracle_1, "Balanced (x₀)"),
    (balanced_oracle_2, "Balanced (x₀⊕x₁)")
]

for ax, (oracle, name) in zip(axes, oracles):
    qc = deutsch_jozsa(3, oracle)
    result = simulator.run(qc, shots=1000).result().get_counts()
    plot_histogram(result, ax=ax, title=name)

plt.tight_layout()
plt.show()
print("Constant → all zeros | Balanced → NOT all zeros")
print("Determined in a SINGLE query!")

---
## 3. Why Does It Work? — Interference

The magic is in **quantum interference**:

1. After the first H gates, all inputs are in superposition
2. The oracle marks inputs with phases: $(-1)^{f(x)}$
3. The final H gates create **constructive** or **destructive** interference
4. For constant $f$: all paths interfere constructively → |0...0>
5. For balanced $f$: |0...0> destructively interferes → never measured

This is our first example of **quantum advantage**: solving a problem exponentially faster than any classical algorithm.

---
## 4. Key Takeaways

| Concept | Description |
|---------|-------------|
| **Oracle** | Black-box function that can be queried in superposition |
| **Phase kickback** | Function value encoded as a phase |
| **Deutsch-Jozsa** | 1 query vs 2^(n-1)+1 queries classically |
| **Interference** | The key to quantum speedup |

---
## 5. Challenges

1. **4-qubit DJ**: Run Deutsch-Jozsa with 4 input qubits. Create a new balanced oracle for it.
2. **Random oracle**: Build an oracle where each CNOT is randomly included. Is it constant or balanced?
3. **Deutsch's algorithm**: Implement the original 1-qubit version (n=1), the simplest case.

In [ ]:
# Your challenge code here!

---
**Next up: [Level 6 — Grover's Algorithm](../Level_06_Grovers_Algorithm/Level_06_Qiskit.ipynb)**